# Agente Gerador de Provas

Agente autônomo que recebe **conteúdo de aula**, **nível de dificuldade** e
**número de questões**, e devolve uma prova completa (múltipla escolha,
dissertação e verdadeiro/falso) com **gabarito comentado** e **critérios de
correção**, aplicando o padrão **Reflection** (o próprio agente avalia a
qualidade de cada questão gerada antes de aceitá-la na prova).

## Estrutura deste notebook
1. Configuração do ambiente (Groq API)
2. Conteúdo de aula de exemplo
3. Funções auxiliares de comunicação com o LLM (saída em JSON)
4. Funções principais do agente:
   - `extrair_topicos(conteudo)`
   - `gerar_multipla_escolha(topico, dificuldade)`
   - `gerar_dissertacao(topico, nivel)`
   - `gerar_vf(topico)`
   - `gerar_gabarito(questao)`
   - `balancear_prova(questoes)`
5. Padrão Reflection (`avaliar_qualidade_questao` + regeneração)
6. Orquestrador `gerar_prova(conteudo, dificuldade, num_questoes)`
7. Formatação de saída (texto legível + JSON estruturado)
8. Teste com os 3 níveis de dificuldade (fácil, médio, difícil)

In [ ]:
!pip install groq -q

from google.colab import userdata

from groq import Groq
import json
import re
import textwrap
from collections import defaultdict
from datetime import datetime

# Chave Groq (salva nos Secrets do Colab)
api_key = userdata.get("GROQ_API_KEY")

if api_key is None:
    raise ValueError("A chave GROQ_API_KEY não foi encontrada. Verifique se ela foi salva nos Secrets do Colab.")

groq_client = Groq(api_key=api_key)
MODEL = "llama-3.1-8b-instant"

print("Ambiente configurado com sucesso!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 436.9 kB/s eta 0:00:00
Ambiente configurado com sucesso!


## 2. Conteúdo de aula de exemplo

Troque o texto abaixo pelo conteúdo real da sua aula (pode ser colado
diretamente, ou você pode adaptar para carregar de um `.txt`/`.pdf`, como no
notebook de RAG). O agente funciona com qualquer conteúdo textual.

In [ ]:
conteudo_aula_exemplo = """
Aula: Arquitetura Cliente-Servidor e APIs REST

Em uma arquitetura cliente-servidor, o cliente (ex: navegador, aplicativo
mobile) envia requisições para um servidor, que processa a requisição e
devolve uma resposta. Essa separação permite escalar o servidor de forma
independente do cliente e reutilizar a mesma API para múltiplos tipos de
cliente.

REST (Representational State Transfer) é um estilo arquitetural para
construção de APIs baseado em recursos. Cada recurso é identificado por uma
URI (ex: /usuarios/42) e manipulado através dos métodos HTTP: GET (ler),
POST (criar), PUT/PATCH (atualizar) e DELETE (remover). Uma API REST deve
ser stateless: o servidor não guarda o estado da sessão do cliente entre
requisições, e cada requisição deve conter todas as informações necessárias
para ser processada.

Os códigos de status HTTP comunicam o resultado da requisição: a faixa 2xx
indica sucesso (200 OK, 201 Created, 204 No Content), a faixa 4xx indica erro
do cliente (400 Bad Request, 401 Unauthorized, 404 Not Found), e a faixa 5xx
indica erro do servidor (500 Internal Server Error).

Autenticação e autorização são frequentemente implementadas com tokens (ex:
JWT), enviados no cabeçalho Authorization das requisições. Autenticação
responde "quem é o usuário?" e autorização responde "o que esse usuário pode
fazer?" — são conceitos relacionados, mas distintos.

Boas práticas de design de API REST incluem: usar substantivos no plural nas
URIs (/pedidos, não /criarPedido), versionar a API (/v1/pedidos), paginar
listas grandes, documentar contratos (ex: OpenAPI/Swagger) e retornar
mensagens de erro padronizadas e informativas.
""".strip()

print(f"Conteúdo carregado: {len(conteudo_aula_exemplo)} caracteres")

## 3. Função auxiliar de comunicação com o LLM

Todas as funções do agente pedem ao modelo uma resposta **estritamente em
JSON**. Como LLMs às vezes retornam markdown ou texto extra em volta do JSON,
`_chamar_llm` extrai o bloco JSON da resposta e faz retentativas automáticas
caso o parse falhe.

In [ ]:
def _extrair_json(texto: str):
    """Remove blocos markdown (```json ... ```) e isola o trecho JSON da resposta."""
    texto = texto.strip()
    texto = re.sub(r"^```json\s*|^```\s*|```$", "", texto, flags=re.MULTILINE).strip()

    inicio_obj, fim_obj = texto.find("{"), texto.rfind("}")
    inicio_lista, fim_lista = texto.find("["), texto.rfind("]")

    # Usa o que aparecer primeiro (objeto {} ou lista []), o que for válido
    if inicio_obj != -1 and (inicio_lista == -1 or inicio_obj < inicio_lista):
        texto = texto[inicio_obj:fim_obj + 1]
    elif inicio_lista != -1:
        texto = texto[inicio_lista:fim_lista + 1]

    return json.loads(texto)


def _chamar_llm(prompt: str, temperature: float = 0.4, max_tokens: int = 800, tentativas: int = 3) -> dict:
    """Chama o modelo Groq e retorna um dict/list já parseado a partir do JSON da resposta.

    Faz até `tentativas` chamadas, reforçando a instrução de formato caso o
    parse do JSON falhe na tentativa anterior.
    """
    ultimo_erro = None
    prompt_atual = prompt

    for _ in range(tentativas):
        r = groq_client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": "Você responde APENAS em JSON válido, em português do Brasil, sem comentários, sem markdown e sem texto fora do JSON."},
                {"role": "user", "content": prompt_atual},
            ],
            temperature=temperature,
            max_tokens=max_tokens,
        )
        conteudo = r.choices[0].message.content
        try:
            return _extrair_json(conteudo)
        except (json.JSONDecodeError, ValueError) as e:
            ultimo_erro = e
            prompt_atual = prompt + (
                "\n\nATENÇÃO: sua resposta anterior não era um JSON válido. "
                "Responda SOMENTE com o JSON solicitado, começando com { ou [."
            )

    raise ValueError(f"Não foi possível obter um JSON válido após {tentativas} tentativas. Último erro: {ultimo_erro}")

## 4. Funções principais do agente

### 4.1 `extrair_topicos(conteudo)`

In [ ]:
def extrair_topicos(conteudo: str, quantidade_maxima: int = 8) -> list:
    """Extrai os principais tópicos de um conteúdo de aula, do mais para o menos relevante."""
    prompt = f"""
Você é um especialista em planejamento pedagógico.
Leia o conteúdo de aula abaixo e extraia os principais tópicos abordados,
do mais para o menos relevante, no máximo {quantidade_maxima} tópicos.

Cada tópico deve ser curto (até 8 palavras) e representar um subtema
específico do conteúdo, não o assunto geral da aula inteira.

Responda SOMENTE com um JSON no formato:
{{"topicos": ["topico 1", "topico 2", "..."]}}

CONTEÚDO DA AULA:
\"\"\"{conteudo}\"\"\"
"""
    resultado = _chamar_llm(prompt, temperature=0.2, max_tokens=400)
    topicos = resultado.get("topicos", [])
    if not topicos:
        raise ValueError("Nenhum tópico foi extraído do conteúdo informado.")
    return topicos

### 4.2 `gerar_multipla_escolha(topico, dificuldade)`

In [ ]:
def gerar_multipla_escolha(topico: str, dificuldade: str) -> dict:
    """Gera uma questão de múltipla escolha (4 alternativas, 1 correta) sobre um tópico."""
    prompt = f"""
Crie UMA questão de múltipla escolha em nível de dificuldade "{dificuldade}"
sobre o tópico: "{topico}".

Regras:
- 4 alternativas (A, B, C, D), apenas uma correta.
- As alternativas incorretas devem ser plausíveis (distratores), não óbvias.
- Não revele a resposta correta no próprio enunciado.

Responda SOMENTE com um JSON no formato:
{{
  "tipo": "multipla_escolha",
  "topico": "{topico}",
  "dificuldade": "{dificuldade}",
  "enunciado": "...",
  "alternativas": {{"A": "...", "B": "...", "C": "...", "D": "..."}},
  "resposta_correta": "A"
}}
"""
    return _chamar_llm(prompt, temperature=0.5, max_tokens=500)

### 4.3 `gerar_dissertacao(topico, nivel)`

In [ ]:
def gerar_dissertacao(topico: str, nivel: str) -> dict:
    """Gera uma questão dissertativa que exige explicação/análise, não só memorização."""
    prompt = f"""
Crie UMA questão dissertativa em nível "{nivel}" sobre o tópico: "{topico}".

A questão deve exigir explicação, análise ou aplicação do conceito, não
apenas a repetição de uma definição decorada.

Responda SOMENTE com um JSON no formato:
{{
  "tipo": "dissertacao",
  "topico": "{topico}",
  "dificuldade": "{nivel}",
  "enunciado": "...",
  "resposta_esperada": "resumo do que se espera na resposta ideal",
  "pontos_chave": ["ponto 1", "ponto 2", "ponto 3"]
}}
"""
    return _chamar_llm(prompt, temperature=0.5, max_tokens=600)

### 4.4 `gerar_vf(topico)`

In [ ]:
def gerar_vf(topico: str, quantidade_afirmativas: int = 4) -> dict:
    """Gera um bloco de afirmativas Verdadeiro/Falso sobre um tópico."""
    prompt = f"""
Crie {quantidade_afirmativas} afirmativas do tipo Verdadeiro/Falso sobre o
tópico: "{topico}".

Regras:
- Misture afirmativas verdadeiras e falsas (pelo menos 1 de cada).
- Afirmativas falsas devem conter um erro sutil e específico, não um absurdo óbvio.

Responda SOMENTE com um JSON no formato:
{{
  "tipo": "verdadeiro_falso",
  "topico": "{topico}",
  "afirmativas": [
    {{"texto": "...", "valor": true}},
    {{"texto": "...", "valor": false}}
  ]
}}
"""
    return _chamar_llm(prompt, temperature=0.5, max_tokens=500)

### 4.5 `gerar_gabarito(questao)`

Gera a resposta comentada / critérios de correção, adaptando o formato ao
`tipo` da questão recebida.

In [ ]:
def gerar_gabarito(questao: dict) -> dict:
    """Gera o gabarito comentado (ou critérios de correção) de uma questão."""
    tipo = questao.get("tipo")

    if tipo == "multipla_escolha":
        prompt = f"""
Questão de múltipla escolha:
Enunciado: {questao['enunciado']}
Alternativas: {json.dumps(questao['alternativas'], ensure_ascii=False)}
Resposta correta: {questao['resposta_correta']}

Escreva uma justificativa comentada explicando por que a alternativa correta
está certa e por que cada uma das outras alternativas está errada (de forma
breve).

Responda SOMENTE com um JSON no formato:
{{
  "resposta_correta": "{questao['resposta_correta']}",
  "justificativa": "...",
  "justificativa_por_alternativa": {{"A": "...", "B": "...", "C": "...", "D": "..."}}
}}
"""

    elif tipo == "dissertacao":
        prompt = f"""
Questão dissertativa:
Enunciado: {questao['enunciado']}
Resposta esperada (resumo): {questao.get('resposta_esperada', '')}
Pontos-chave: {questao.get('pontos_chave', [])}

Crie critérios de correção detalhados, com pontuação sugerida somando 10
pontos, distribuída entre os pontos-chave.

Responda SOMENTE com um JSON no formato:
{{
  "criterios_correcao": [{{"criterio": "...", "pontos": 0}}],
  "resposta_modelo": "..."
}}
"""

    elif tipo == "verdadeiro_falso":
        prompt = f"""
Afirmativas verdadeiro/falso: {json.dumps(questao['afirmativas'], ensure_ascii=False)}

Para cada afirmativa, escreva uma justificativa curta explicando por que ela
é verdadeira ou falsa.

Responda SOMENTE com um JSON no formato:
{{"justificativas": [{{"texto": "...", "valor": true, "justificativa": "..."}}]}}
"""

    else:
        raise ValueError(f"Tipo de questão desconhecido: {tipo}")

    return _chamar_llm(prompt, temperature=0.2, max_tokens=700)

### 4.6 `balancear_prova(questoes)`

Reordena as questões geradas para intercalar tópicos (evita que a prova
comece com 3 questões do mesmo tópico seguidas) e reporta a distribuição
final por tópico e por tipo.

In [ ]:
def balancear_prova(questoes: list) -> list:
    """Reordena as questões intercalando tópicos, para uma cobertura mais equilibrada.

    Não descarta nem gera questões novas: apenas reorganiza a ordem final da
    prova para que tópicos e tipos fiquem bem distribuídos ao longo dela.
    """
    por_topico = defaultdict(list)
    for q in questoes:
        por_topico[q.get("topico", "geral")].append(q)

    topicos_na_ordem_original = list(dict.fromkeys(q.get("topico", "geral") for q in questoes))
    max_rodadas = max((len(v) for v in por_topico.values()), default=0)

    balanceada = []
    for rodada in range(max_rodadas):
        for topico in topicos_na_ordem_original:
            if rodada < len(por_topico[topico]):
                balanceada.append(por_topico[topico][rodada])

    return balanceada


def distribuicao_da_prova(questoes: list) -> dict:
    """Retorna um resumo da distribuição de questões por tópico e por tipo (útil para conferência)."""
    por_topico = defaultdict(int)
    por_tipo = defaultdict(int)
    for q in questoes:
        por_topico[q.get("topico", "geral")] += 1
        por_tipo[q.get("tipo", "desconhecido")] += 1
    return {"por_topico": dict(por_topico), "por_tipo": dict(por_tipo)}

## 5. Padrão Reflection

Depois de gerar uma questão, o próprio agente assume o papel de **revisor
pedagógico** e avalia a qualidade da questão (clareza, adequação ao nível,
qualidade dos distratores, ausência de erros factuais). Se a nota ficar
abaixo do limite de aprovação, a questão é **regenerada** (até
`max_tentativas`), e a melhor versão entre as tentativas é mantida.

In [ ]:
def avaliar_qualidade_questao(questao: dict) -> dict:
    """Avalia (nota 0-10) a qualidade de uma questão gerada. Este é o passo de 'reflexão' do agente."""
    prompt = f"""
Você é um revisor pedagógico rigoroso. Avalie a qualidade da questão abaixo.

QUESTÃO:
{json.dumps(questao, ensure_ascii=False, indent=2)}

Critérios de avaliação (nota de 0 a 10):
- Clareza do enunciado (sem ambiguidade)
- Adequação ao nível de dificuldade declarado
- Se for múltipla escolha: distratores plausíveis e resposta correta inequívoca
- Se for dissertativa: exige análise/aplicação, não só memorização
- Se for V/F: erros sutis nas afirmativas falsas, não absurdos óbvios
- Ausência de erros factuais

Responda SOMENTE com um JSON no formato:
{{
  "nota": 0,
  "pontos_fortes": ["..."],
  "pontos_fracos": ["..."],
  "aprovada": true
}}
Considere aprovada = true apenas se nota >= 7.
"""
    return _chamar_llm(prompt, temperature=0.1, max_tokens=500)


_GERADORES_POR_TIPO = {
    "multipla_escolha": lambda topico, dificuldade: gerar_multipla_escolha(topico, dificuldade),
    "dissertacao": lambda topico, dificuldade: gerar_dissertacao(topico, dificuldade),
    "verdadeiro_falso": lambda topico, dificuldade: gerar_vf(topico),
}


def gerar_questao_com_reflexao(tipo: str, topico: str, dificuldade: str, max_tentativas: int = 2) -> dict:
    """Gera uma questão e aplica o padrão Reflection: avalia, e regenera se reprovada.

    Mantém a melhor tentativa (maior nota) entre `max_tentativas` gerações,
    mesmo que nenhuma atinja a nota de aprovação.
    """
    melhor_questao, melhor_avaliacao = None, None

    for tentativa in range(1, max_tentativas + 1):
        questao = _GERADORES_POR_TIPO[tipo](topico, dificuldade)
        avaliacao = avaliar_qualidade_questao(questao)
        questao["_avaliacao"] = {**avaliacao, "tentativa": tentativa}

        if melhor_avaliacao is None or avaliacao.get("nota", 0) > melhor_avaliacao.get("nota", 0):
            melhor_questao, melhor_avaliacao = questao, avaliacao

        if avaliacao.get("aprovada"):
            break

    return melhor_questao

## 6. Orquestrador — `gerar_prova(conteudo, dificuldade, num_questoes)`

Une todas as etapas: extrai tópicos → gera questões intercalando os 3 tipos
(múltipla escolha, dissertação, V/F) com Reflection → balanceia a prova →
gera o gabarito comentado de cada questão.

In [ ]:
def gerar_prova(conteudo: str, dificuldade: str, num_questoes: int) -> dict:
    """Agente autônomo: gera uma prova completa a partir de um conteúdo de aula.

    Parâmetros
    ----------
    conteudo : str
        Conteúdo da aula (texto corrido).
    dificuldade : str
        "facil", "medio" ou "dificil".
    num_questoes : int
        Número total de questões da prova (mínimo 3, para cobrir os 3 tipos).
    """
    if dificuldade not in ("facil", "medio", "dificil"):
        raise ValueError('dificuldade deve ser "facil", "medio" ou "dificil"')
    if num_questoes < 3:
        raise ValueError("num_questoes deve ser pelo menos 3, para cobrir os 3 tipos de questão")

    topicos = extrair_topicos(conteudo)

    tipos_disponiveis = ["multipla_escolha", "dissertacao", "verdadeiro_falso"]
    plano_de_tipos = [tipos_disponiveis[i % len(tipos_disponiveis)] for i in range(num_questoes)]

    questoes = []
    for i, tipo in enumerate(plano_de_tipos):
        topico = topicos[i % len(topicos)]
        questao = gerar_questao_com_reflexao(tipo, topico, dificuldade)
        questoes.append(questao)

    questoes = balancear_prova(questoes)

    for questao in questoes:
        questao["gabarito"] = gerar_gabarito(questao)

    return {
        "metadados": {
            "dificuldade": dificuldade,
            "num_questoes": len(questoes),
            "topicos_identificados": topicos,
            "distribuicao": distribuicao_da_prova(questoes),
            "gerado_em": datetime.now().isoformat(timespec="seconds"),
        },
        "questoes": questoes,
    }

## 7. Formatação de saída (texto legível)

A prova é sempre retornada como um **dict Python / JSON** por `gerar_prova`.
`formatar_prova_texto` converte esse JSON em um texto legível, pronto para
impressão ou exportação, com a prova primeiro e o gabarito comentado
depois.

In [ ]:
def formatar_prova_texto(prova: dict) -> str:
    """Converte o JSON da prova em um texto legível (prova + gabarito comentado)."""
    meta = prova["metadados"]
    linhas = []

    linhas.append("=" * 78)
    linhas.append(f"PROVA — Nível: {meta['dificuldade'].upper()} — {meta['num_questoes']} questões")
    linhas.append("=" * 78)
    linhas.append(f"Tópicos identificados: {', '.join(meta['topicos_identificados'])}")

    for i, questao in enumerate(prova["questoes"], start=1):
        linhas.append(f"\nQuestão {i} [{questao['tipo']}] — Tópico: {questao.get('topico', '-')}")
        linhas.append("-" * 78)

        if questao["tipo"] == "multipla_escolha":
            linhas.append(textwrap.fill(questao["enunciado"], width=78))
            for letra, texto in questao["alternativas"].items():
                linhas.append(f"  {letra}) {texto}")

        elif questao["tipo"] == "dissertacao":
            linhas.append(textwrap.fill(questao["enunciado"], width=78))

        elif questao["tipo"] == "verdadeiro_falso":
            for j, af in enumerate(questao["afirmativas"], start=1):
                linhas.append(f"  ({j}) {af['texto']}   [ V ]  [ F ]")

    linhas.append("\n" + "=" * 78)
    linhas.append("GABARITO COMENTADO")
    linhas.append("=" * 78)

    for i, questao in enumerate(prova["questoes"], start=1):
        gab = questao["gabarito"]
        linhas.append(f"\nQuestão {i} [{questao['tipo']}]")

        if questao["tipo"] == "multipla_escolha":
            linhas.append(f"Resposta correta: {gab['resposta_correta']}")
            linhas.append(textwrap.fill(f"Justificativa: {gab['justificativa']}", width=78))

        elif questao["tipo"] == "dissertacao":
            linhas.append("Critérios de correção:")
            for c in gab.get("criterios_correcao", []):
                linhas.append(f"  - {c['criterio']} ({c['pontos']} pts)")
            linhas.append(textwrap.fill(f"Resposta modelo: {gab.get('resposta_modelo', '')}", width=78))

        elif questao["tipo"] == "verdadeiro_falso":
            for j in gab.get("justificativas", []):
                valor = "V" if j["valor"] else "F"
                linhas.append(textwrap.fill(f"  ({valor}) {j['texto']} — {j['justificativa']}", width=78))

    return "\n".join(linhas)

## 8. Teste com os 3 níveis de dificuldade

Gera uma prova completa para cada nível (fácil, médio, difícil) a partir do
mesmo conteúdo de aula, imprime a versão em texto e salva tudo em
`provas_geradas.json` (saída estruturada em JSON).

In [ ]:
provas_geradas = {}

for nivel in ["facil", "medio", "dificil"]:
    print(f"\n>>> Gerando prova nível: {nivel} ...")
    prova = gerar_prova(conteudo_aula_exemplo, dificuldade=nivel, num_questoes=6)
    provas_geradas[nivel] = prova
    print(formatar_prova_texto(prova))
    print()

with open("provas_geradas.json", "w", encoding="utf-8") as f:
    json.dump(provas_geradas, f, ensure_ascii=False, indent=2)

print("Arquivo provas_geradas.json salvo com sucesso!")

## 9. Inspeção rápida da saída em JSON

Exemplo de como consumir a saída estruturada (por exemplo, para alimentar
uma interface web ou um app de correção automática).

In [ ]:
print(json.dumps(provas_geradas["medio"]["questoes"][0], ensure_ascii=False, indent=2))